In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *



In [4]:
from pyspark.sql import SparkSession

# Initialize your Spark instance
spark = SparkSession.builder \
    .appName("MySparkApp") \
    .getOrCreate()


In [5]:
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("BigMart Sales.csv")
)

In [6]:
df.show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Superma

In [7]:
df.count()

8523

In [8]:
df.head()
# df.tail()

Row(Item_Identifier='FDA15', Item_Weight=9.3, Item_Fat_Content='Low Fat', Item_Visibility=0.016047301, Item_Type='Dairy', Item_MRP=249.8092, Outlet_Identifier='OUT049', Outlet_Establishment_Year=1999, Outlet_Size='Medium', Outlet_Location_Type='Tier 1', Outlet_Type='Supermarket Type1', Item_Outlet_Sales=3735.138)

In [9]:
df.printSchema()

root
 |-- Item_Identifier: string (nullable = true)
 |-- Item_Weight: double (nullable = true)
 |-- Item_Fat_Content: string (nullable = true)
 |-- Item_Visibility: double (nullable = true)
 |-- Item_Type: string (nullable = true)
 |-- Item_MRP: double (nullable = true)
 |-- Outlet_Identifier: string (nullable = true)
 |-- Outlet_Establishment_Year: integer (nullable = true)
 |-- Outlet_Size: string (nullable = true)
 |-- Outlet_Location_Type: string (nullable = true)
 |-- Outlet_Type: string (nullable = true)
 |-- Item_Outlet_Sales: double (nullable = true)



In [10]:
df = (
    df.withColumn(
        "current_date_col",
        current_date()
    )
    .withColumn(
        "week_after_current_date",
        date_add(current_date(), 7)
    )
)

In [11]:
df.limit(5).show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----------------+-----------------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|current_date_col|week_after_current_date|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----------------+-----------------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|      2026-06-10|             2026-06-17|
|          D

In [12]:
df.printSchema()

root
 |-- Item_Identifier: string (nullable = true)
 |-- Item_Weight: double (nullable = true)
 |-- Item_Fat_Content: string (nullable = true)
 |-- Item_Visibility: double (nullable = true)
 |-- Item_Type: string (nullable = true)
 |-- Item_MRP: double (nullable = true)
 |-- Outlet_Identifier: string (nullable = true)
 |-- Outlet_Establishment_Year: integer (nullable = true)
 |-- Outlet_Size: string (nullable = true)
 |-- Outlet_Location_Type: string (nullable = true)
 |-- Outlet_Type: string (nullable = true)
 |-- Item_Outlet_Sales: double (nullable = true)
 |-- current_date_col: date (nullable = false)
 |-- week_after_current_date: date (nullable = false)



In [13]:
df.dropna(subset=['Outlet_Size']).show(5)

+---------------+-----------+----------------+---------------+------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----------------+-----------------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|   Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|current_date_col|week_after_current_date|
+---------------+-----------+----------------+---------------+------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----------------+-----------------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|       Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|      2026-06-10|             2026-06-17|
|          DRC01|       5.92|         Regula

In [14]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')).show(5)
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[0]).show(5)

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+--------------------+-----------------+----------------+-----------------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|         Outlet_Type|Item_Outlet_Sales|current_date_col|week_after_current_date|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+--------------------+-----------------+----------------+-----------------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|[Supermarket, Type1]|         3735.138|      2026-06-10|             2026-06-17|


In [15]:
# df.groupBy('Item_Type').agg(sum('Item_MRP')).show(5)
# df.groupBy('Item_Type').agg(mean('Item_MRP')).show(5)
df.groupBy('Item_Type').agg(
  sum('Item_MRP').alias('SUM_AGG'),
  mean('Item_MRP').alias('AVG_AGG'),
  min('Item_MRP').alias('MIN_AGG'),
  max('Item_MRP').alias('MAX_AGG'),
  count('*').alias('COUNT_AGG')
).show(5)

+--------------------+------------------+------------------+-------+--------+---------+
|           Item_Type|           SUM_AGG|           AVG_AGG|MIN_AGG| MAX_AGG|COUNT_AGG|
+--------------------+------------------+------------------+-------+--------+---------+
|       Starchy Foods|21880.027399999995|147.83802297297294|34.0532|263.0252|      148|
|        Baking Goods| 81894.73640000001|126.38076604938273|33.9874|265.5568|      648|
|              Breads| 35379.11979999999| 140.9526685258964|31.9558|263.6594|      251|
|Fruits and Vegeta...|178124.08099999998|144.58123457792206|36.2506|264.2252|     1232|
|                Meat|59449.863799999956|139.88203247058814|34.7532|261.5936|      425|
+--------------------+------------------+------------------+-------+--------+---------+
only showing top 5 rows


In [16]:
df.groupBy('Item_Type','Outlet_Size').agg(
  sum('Item_MRP').alias('SUM_AGG'),
  avg('Item_MRP').alias('AVG_AGG'),
  std('Item_MRP').alias('STD_AGG')
).show(5)

+--------------------+-----------+------------------+------------------+-----------------+
|           Item_Type|Outlet_Size|           SUM_AGG|           AVG_AGG|          STD_AGG|
+--------------------+-----------+------------------+------------------+-----------------+
|       Starchy Foods|     Medium| 7124.136199999997| 148.4195041666666|67.64107219898328|
|Fruits and Vegeta...|     Medium|59047.217200000014| 142.9714702179177| 60.6156076506207|
|       Starchy Foods|       NULL|         6040.6402|140.48000465116277|74.99667347394231|
|              Breads|       NULL|        10011.5004|139.04861666666667|64.38289263727012|
|        Baking Goods|       NULL|23433.838799999994|126.66939891891889|57.54353384412345|
+--------------------+-----------+------------------+------------------+-----------------+
only showing top 5 rows


In [17]:
df.groupBy('Item_Type').agg(
  collect_list('Item_Fat_Content').alias('COLLECT_LIST_AGG'),
  collect_set('Item_Fat_Content').alias('COLLECT_SET_AGG')
).show(5)

+--------------------+--------------------+--------------------+
|           Item_Type|    COLLECT_LIST_AGG|     COLLECT_SET_AGG|
+--------------------+--------------------+--------------------+
|       Starchy Foods|[Low Fat, Low Fat...|[Low Fat, Regular...|
|        Baking Goods|[Regular, Regular...|[Low Fat, Regular...|
|              Breads|[Low Fat, Regular...|[Low Fat, Regular...|
|Fruits and Vegeta...|[Regular, Low Fat...|[Low Fat, Regular...|
|                Meat|[Low Fat, Low Fat...|[Low Fat, Regular...|
+--------------------+--------------------+--------------------+
only showing top 5 rows


In [18]:
df.groupBy('Item_Type').\
  pivot('Outlet_Size').\
  agg(avg('Item_MRP')).show(5)

+--------------------+------------------+------------------+------------------+------------------+
|           Item_Type|              null|              High|            Medium|             Small|
+--------------------+------------------+------------------+------------------+------------------+
|       Starchy Foods|140.48000465116277|158.15707368421053| 148.4195041666666| 150.2701736842105|
|              Breads|139.04861666666667|         133.75896| 140.8610385542169| 145.5236507042254|
|        Baking Goods|126.66939891891889|129.20204383561642|126.17856847290639|125.21336363636368|
|Fruits and Vegeta...|142.57516045845267|145.57287042253515| 142.9714702179177|148.31336951219507|
|                Meat|139.29453448275865| 137.2447902439025|136.41913154362408|145.69925042016808|
+--------------------+------------------+------------------+------------------+------------------+
only showing top 5 rows


In [19]:
df=df.withColumn("food_category",when(col("Item_Type").isin("Meat","Seafood"), "Non-Veg").otherwise("Veg"))

In [20]:
df.show(5)

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----------------+-----------------------+-------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|current_date_col|week_after_current_date|food_category|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----------------+-----------------------+-------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|      2026-0

In [21]:
df.withColumn('food_category_expense',when(
  (col('food_category')=='Veg') & (col('Item_MRP')<100),'sasta_hai'
  ).otherwise('expensive')).show(5)

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----------------+-----------------------+-------------+---------------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|current_date_col|week_after_current_date|food_category|food_category_expense|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+----------------+-----------------------+-------------+---------------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|   

In [22]:
import os

print(os.environ.get("PYSPARK_PYTHON"))
print(os.environ.get("PYSPARK_DRIVER_PYTHON"))

C:\Python313\python.exe
C:\Python313\python.exe


In [23]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [24]:
df1.show()
df2.show()

+------+--------+-------+
|emp_id|emp_name|dept_id|
+------+--------+-------+
|     1|    gaur|    d01|
|     2|     kit|    d02|
|     3|     sam|    d03|
|     4|     tim|    d03|
|     5|    aman|    d05|
|     6|     nad|    d06|
+------+--------+-------+

+-------+----------+
|dept_id|department|
+-------+----------+
|    d01|        HR|
|    d02| Marketing|
|    d03|  Accounts|
|    d04|        IT|
|    d05|   Finance|
+-------+----------+



In [25]:
from pyspark.sql.window import Window